# XGBoost Ablation Study — Grid-Search-Aligned

This notebook runs the XGBoost ablation (Full / No ISP / No BM / No ISP+BM) using the **exact same grid search and training procedure** as the main model comparison notebooks (B and C).

**Key difference from previous ablation:** The best hyperparameters are found via grid search on the Full feature set, and then the **same best params** are used for all 4 feature set variants. This ensures the "Full" baseline matches the main comparison table exactly.

**How to use:** Set `PERIOD` to `'B'` or `'C'` in the config cell below.

In [1]:
!pip install xgboost -q

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ═══════════════════════════════════════════════════════════
# CONFIG — set period here
# ═══════════════════════════════════════════════════════════
PERIOD = 'C'  # 'B' for Full sample, 'C' for Spring sample

if PERIOD == 'B':
    DATE_START = '2025-10-02'
    DATE_END   = '2026-03-12 23:45:00'
    PERIOD_NAME = 'Period B: Full sample'
elif PERIOD == 'C':
    DATE_START = '2026-01-20'
    DATE_END   = '2026-03-12 23:45:00'
    PERIOD_NAME = 'Period C: Spring sample'
else:
    raise ValueError(f"Unknown period: {PERIOD}")

print(f"Running ablation for {PERIOD_NAME}")

Running ablation for Period C: Spring sample


## 1. Data Loading & Feature Engineering

Identical to the main comparison notebooks (B/C).

In [2]:
FILENAME = '/content/Final2026_with_ENTSOE.csv'

try:
    df = pd.read_csv(FILENAME, sep=',')
    if len(df.columns) == 1: df = pd.read_csv(FILENAME, sep=';')
except: df = pd.read_csv(FILENAME)

df = df.rename(columns={
    'Day time': 'DELIVERY_MTU',
    'DAM price': 'DAM_MCP',
    'IDA1': 'MCP',
    'IDA2': 'MCP_IDA2',
    'IDA3': 'MCP_IDA3',
    'BM price': 'BM_IMBALANCE_PRICE',
    'BMmDAM price': 'BMmDAMMCP',
})

df['DELIVERY_MTU'] = pd.to_datetime(df['DELIVERY_MTU'])
df = df.sort_values('DELIVERY_MTU').reset_index(drop=True)

# Time features
df['hour']      = df['DELIVERY_MTU'].dt.hour
df['minute']    = df['DELIVERY_MTU'].dt.minute / 60.0
df['dayofweek'] = df['DELIVERY_MTU'].dt.dayofweek
df['ida3_available'] = df['MCP_IDA3'].notna().astype(float)

# Fill missing
df['MCP_IDA3']  = df['MCP_IDA3'].fillna(0.0)
df['MCP_IDA2']  = df['MCP_IDA2'].ffill().bfill().fillna(0.0)
df['MCP']       = df['MCP'].ffill().bfill().fillna(df['MCP_IDA2'])
df['DAM_MCP']   = df['DAM_MCP'].ffill().bfill()
df['BMmDAMMCP'] = df['BMmDAMMCP'].ffill().bfill().fillna(0.0)
df['BM_IMBALANCE_PRICE'] = df['BM_IMBALANCE_PRICE'].ffill().bfill().fillna(df['BM_IMBALANCE_PRICE'].median())
df['IDA_target'] = np.where(df['ida3_available'] == 1, df['MCP_IDA3'], df['MCP_IDA2'])
df['ISP2_RES']  = df['ISP2_RES'].ffill().bfill().fillna(df['ISP1_RES'])
df['ISP2_Load'] = df['ISP2_Load'].ffill().bfill().fillna(df['ISP1_Load'])

# ISP revision features
df['DA_RES'] = df['Wind_DA_forecast'] + df['Solar_DA_forecast']
is_ida3 = df['hour'] >= 12
df['latest_RES']  = np.where(is_ida3, df['ISP3_RES'], df['ISP2_RES'])
df['latest_Load'] = np.where(is_ida3, df['ISP3_Load'], df['ISP2_Load'])
df['latest_RES_revision']  = df['latest_RES'] - df['DA_RES']
df['latest_Load_revision'] = df['latest_Load'] - df['SystemLoad_DA_forecast']
df['latest_net_load']      = df['latest_Load'] - df['latest_RES']
df['DA_net_load']          = df['SystemLoad_DA_forecast'] - df['DA_RES']
df['latest_net_load_revision'] = df['latest_net_load'] - df['DA_net_load']
df['RES_revision_ISP2_to_ISP3']  = np.where(is_ida3, df['ISP3_RES'] - df['ISP2_RES'], 0.0)
df['Load_revision_ISP2_to_ISP3'] = np.where(is_ida3, df['ISP3_Load'] - df['ISP2_Load'], 0.0)

# BM and momentum features
df['bm_roll_mean_4']  = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).mean()
df['bm_roll_std_4']   = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).std().fillna(0)
df['bm_roll_mean_12'] = df['BM_IMBALANCE_PRICE'].rolling(12, min_periods=1).mean()
df['ida_momentum']  = df['MCP_IDA2'].diff(4).fillna(0)
df['mcp_momentum']  = df['MCP'].diff(4).fillna(0)
df['bm_momentum']   = df['BM_IMBALANCE_PRICE'].diff(4).fillna(0)
df['bm_ida_spread']  = df['BM_IMBALANCE_PRICE'] - df['MCP_IDA2']
df['ida_roll_std_4'] = df['MCP_IDA2'].rolling(4, min_periods=1).std().fillna(0)

# Targets
for h_steps, h_name in [(4, '1h'), (8, '2h'), (12, '3h')]:
    df[f'IDA_target_{h_name}'] = df['IDA_target'].shift(-h_steps)
    df[f'ida3_at_{h_name}'] = (df['hour'].shift(-h_steps) >= 12).astype(float)

df = df.dropna(subset=['IDA_target_1h', 'IDA_target_2h', 'IDA_target_3h']).reset_index(drop=True)

# Date filter
df = df[df['DELIVERY_MTU'] >= DATE_START].copy().reset_index(drop=True)
df = df[df['DELIVERY_MTU'] <= DATE_END].copy().reset_index(drop=True)

# IDA-DAM spread (for diagnostics only)
df['IDA_DAM_spread'] = df['IDA_target'] - df['DAM_MCP']

TARGETS = ['IDA_target_1h', 'IDA_target_2h', 'IDA_target_3h']
HORIZON = 3
TRAIN_RATIO = 0.8
split_idx = int(len(df) * TRAIN_RATIO)

# Fix any remaining NaNs
all_feats = [
    'MCP', 'MCP_IDA2', 'DAM_MCP',
    'BM_IMBALANCE_PRICE', 'bm_roll_mean_4', 'bm_roll_std_4', 'bm_roll_mean_12', 'bm_ida_spread',
    'ida_momentum', 'mcp_momentum', 'bm_momentum', 'ida_roll_std_4',
    'SystemLoad_DA_forecast', 'Wind_DA_forecast', 'Solar_DA_forecast',
    'ida3_available', 'ida3_at_1h', 'ida3_at_2h', 'ida3_at_3h',
    'hour', 'minute', 'dayofweek',
    'latest_RES', 'latest_Load',
    'latest_RES_revision', 'latest_Load_revision', 'latest_net_load_revision', 'latest_net_load',
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]
nan_check = df[all_feats].isna().sum()
if nan_check.sum() > 0:
    print("WARNING: NaN in features:")
    print(nan_check[nan_check > 0])
    df[all_feats] = df[all_feats].ffill().bfill().fillna(0)
    print("Fixed with ffill/bfill/fillna(0)")

print(f"Dataset: {df.shape}")
print(f"Train: {split_idx}, Test: {len(df)-split_idx}")
print(f"Period: {df['DELIVERY_MTU'].min()} to {df['DELIVERY_MTU'].max()}")

Dataset: (4992, 46)
Train: 3993, Test: 999
Period: 2026-01-20 00:00:00 to 2026-03-12 23:45:00


## 2. Feature Set Definitions

Same 4 feature groups as the main comparison notebooks.

In [3]:
FEATURES_FULL = [
    # Prices (3)
    'MCP', 'MCP_IDA2', 'DAM_MCP',
    # BM signal (5)
    'BM_IMBALANCE_PRICE', 'bm_roll_mean_4', 'bm_roll_std_4', 'bm_roll_mean_12', 'bm_ida_spread',
    # Momentum (4)
    'ida_momentum', 'mcp_momentum', 'bm_momentum', 'ida_roll_std_4',
    # DA context (3)
    'SystemLoad_DA_forecast', 'Wind_DA_forecast', 'Solar_DA_forecast',
    # Auction flags (4)
    'ida3_available', 'ida3_at_1h', 'ida3_at_2h', 'ida3_at_3h',
    # Time (3)
    'hour', 'minute', 'dayofweek',
    # ISP raw (2)
    'latest_RES', 'latest_Load',
    # ISP revisions (4)
    'latest_RES_revision', 'latest_Load_revision', 'latest_net_load_revision', 'latest_net_load',
    # ISP sequential (2)
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]

ISP_FEATURES = [
    'latest_RES', 'latest_Load',
    'latest_RES_revision', 'latest_Load_revision',
    'latest_net_load_revision', 'latest_net_load',
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]

BM_FEATURES = [
    'BM_IMBALANCE_PRICE', 'bm_roll_mean_4', 'bm_roll_std_4',
    'bm_roll_mean_12', 'bm_ida_spread', 'bm_momentum',
]

FEATURES_NO_ISP  = [f for f in FEATURES_FULL if f not in ISP_FEATURES]
FEATURES_NO_BM   = [f for f in FEATURES_FULL if f not in BM_FEATURES]
FEATURES_NO_BOTH = [f for f in FEATURES_FULL if f not in ISP_FEATURES + BM_FEATURES]

feature_sets = {
    'Full (30 feat)':      FEATURES_FULL,
    'No ISP (22 feat)':    FEATURES_NO_ISP,
    'No BM (24 feat)':     FEATURES_NO_BM,
    'No ISP+BM (16 feat)': FEATURES_NO_BOTH,
}

for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

  Full (30 feat): 30 features
  No ISP (22 feat): 22 features
  No BM (24 feat): 24 features
  No ISP+BM (16 feat): 16 features


## 3. XGBoost Grid Search on Full Features

**Identical** grid search to the main comparison notebooks:
- Lookback: [24, 48]
- Max depth: [4, 6]
- N estimators: [200, 400]
- Learning rate: [0.05, 0.1]
- Fixed: subsample=0.8, colsample_bytree=0.8, objective='reg:absoluteerror'

The best params found here will be used for **all 4 ablation variants**.

In [4]:
def build_xgb_data(features, lookback):
    """Build flattened lookback windows — identical to main notebooks."""
    data = df[features].values.astype(np.float32)
    targets = df[TARGETS].values.astype(np.float32)
    n = len(data) - lookback
    X = np.zeros((n, lookback * len(features)), dtype=np.float32)
    y = np.zeros((n, HORIZON), dtype=np.float32)
    for i in range(n):
        X[i] = data[i:i+lookback].flatten()
        y[i] = targets[i + lookback]
    s = split_idx - lookback
    return X[:s], X[s:], y[:s], y[s:]


# ── Grid search on FULL features (same grid as main notebooks) ──
print(f'XGBoost Grid Search on Full features ({PERIOD_NAME})...')
print(f'  Grid: lookback=[24,48] x depth=[4,6] x trees=[200,400] x lr=[0.05,0.1]')
print(f'  Total configs: 16\n')

best_mae = float('inf')
bp_xgb = {'lookback': 24, 'max_depth': 4, 'n_estimators': 200,
           'learning_rate': 0.05, 'subsample': 0.8}

for lb in [24, 48]:
    for depth in [4, 6]:
        for n_est in [200, 400]:
            for lr in [0.05, 0.1]:
                X_tr, X_te, y_tr, y_te = build_xgb_data(FEATURES_FULL, lb)
                m = xgb.XGBRegressor(
                    max_depth=depth, n_estimators=n_est,
                    learning_rate=lr, subsample=0.8, colsample_bytree=0.8,
                    objective='reg:absoluteerror', tree_method='hist',
                    n_jobs=-1, verbosity=0)
                m.fit(X_tr, y_tr[:, 0],
                      eval_set=[(X_te, y_te[:, 0])], verbose=False)
                mae = mean_absolute_error(y_te[:, 0], m.predict(X_te))
                if mae < best_mae:
                    best_mae = mae
                    bp_xgb = {'lookback': lb, 'max_depth': depth,
                              'n_estimators': n_est, 'learning_rate': lr,
                              'subsample': 0.8}
                    print(f'  New best: lb={lb} depth={depth} trees={n_est} lr={lr} MAE={mae:.2f}')

print(f'\n  Best XGBoost params: {bp_xgb}')
print(f'  These params will be used for ALL 4 ablation variants.')

XGBoost Grid Search on Full features (Period C: Spring sample)...
  Grid: lookback=[24,48] x depth=[4,6] x trees=[200,400] x lr=[0.05,0.1]
  Total configs: 16

  New best: lb=24 depth=4 trees=200 lr=0.05 MAE=25.32
  New best: lb=24 depth=4 trees=400 lr=0.05 MAE=25.17

  Best XGBoost params: {'lookback': 24, 'max_depth': 4, 'n_estimators': 400, 'learning_rate': 0.05, 'subsample': 0.8}
  These params will be used for ALL 4 ablation variants.


## 4. Ablation Study

Train XGBoost with the **best params from grid search** on each feature set variant.
Includes `reg_alpha=0.1, reg_lambda=1.0` (same as the final training cell in the main notebooks).

One separate model per horizon (same as main notebooks).

In [5]:
horizon_names = ['1h', '2h', '3h']
ablation_results = {}

print(f'Running ablation with best params: {bp_xgb}')
print(f'Additional: reg_alpha=0.1, reg_lambda=1.0, colsample_bytree=0.8')
print(f'{"=" * 65}\n')

for set_name, feats in feature_sets.items():
    lb = bp_xgb['lookback']
    X_tr, X_te, y_tr, y_te = build_xgb_data(feats, lb)
    maes = []
    for h in range(HORIZON):
        m = xgb.XGBRegressor(
            max_depth=bp_xgb['max_depth'],
            n_estimators=bp_xgb['n_estimators'],
            learning_rate=bp_xgb['learning_rate'],
            subsample=bp_xgb['subsample'],
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective='reg:absoluteerror',
            tree_method='hist',
            n_jobs=-1, verbosity=0)
        m.fit(X_tr, y_tr[:, h],
              eval_set=[(X_te, y_te[:, h])], verbose=False)
        mae = mean_absolute_error(y_te[:, h], m.predict(X_te))
        maes.append(mae)
    ablation_results[set_name] = maes
    print(f'  {set_name:<22}  t+1h={maes[0]:.2f}  t+2h={maes[1]:.2f}  t+3h={maes[2]:.2f}')

print()

Running ablation with best params: {'lookback': 24, 'max_depth': 4, 'n_estimators': 400, 'learning_rate': 0.05, 'subsample': 0.8}
Additional: reg_alpha=0.1, reg_lambda=1.0, colsample_bytree=0.8

  Full (30 feat)          t+1h=25.19  t+2h=30.28  t+3h=36.84
  No ISP (22 feat)        t+1h=25.59  t+2h=31.52  t+3h=38.53
  No BM (24 feat)         t+1h=25.14  t+2h=31.58  t+3h=36.98
  No ISP+BM (16 feat)     t+1h=26.14  t+2h=32.48  t+3h=38.05



## 5. Results Table

In [6]:
print(f'\n{"=" * 65}')
print(f'  XGBoost Ablation — {PERIOD_NAME}')
print(f'{"=" * 65}')
print(f'{"Feature Set":<24} {"MAE (1h)":>10} {"MAE (2h)":>10} {"MAE (3h)":>10}')
print(f'{"-" * 54}')
for name, maes in ablation_results.items():
    print(f'{name:<24} {maes[0]:>10.2f} {maes[1]:>10.2f} {maes[2]:>10.2f}')

# ISP and BM contribution
full = ablation_results['Full (30 feat)']
no_isp = ablation_results['No ISP (22 feat)']
no_bm = ablation_results['No BM (24 feat)']
no_both = ablation_results['No ISP+BM (16 feat)']

print(f'\n  ISP contribution (MAE reduction vs No ISP):')
for h in range(HORIZON):
    delta = no_isp[h] - full[h]
    pct = delta / no_isp[h] * 100
    print(f'    t+{horizon_names[h]}: {delta:+.2f} EUR/MWh ({pct:+.1f}%)')

print(f'\n  BM contribution (MAE reduction vs No BM):')
for h in range(HORIZON):
    delta = no_bm[h] - full[h]
    pct = delta / no_bm[h] * 100
    print(f'    t+{horizon_names[h]}: {delta:+.2f} EUR/MWh ({pct:+.1f}%)')

print(f'\n  Best XGBoost params used: {bp_xgb}')
print(f'  reg_alpha=0.1, reg_lambda=1.0 (same as main comparison notebook)')


  XGBoost Ablation — Period C: Spring sample
Feature Set                MAE (1h)   MAE (2h)   MAE (3h)
------------------------------------------------------
Full (30 feat)                25.19      30.28      36.84
No ISP (22 feat)              25.59      31.52      38.53
No BM (24 feat)               25.14      31.58      36.98
No ISP+BM (16 feat)           26.14      32.48      38.05

  ISP contribution (MAE reduction vs No ISP):
    t+1h: +0.40 EUR/MWh (+1.6%)
    t+2h: +1.24 EUR/MWh (+3.9%)
    t+3h: +1.69 EUR/MWh (+4.4%)

  BM contribution (MAE reduction vs No BM):
    t+1h: -0.05 EUR/MWh (-0.2%)
    t+2h: +1.30 EUR/MWh (+4.1%)
    t+3h: +0.14 EUR/MWh (+0.4%)

  Best XGBoost params used: {'lookback': 24, 'max_depth': 4, 'n_estimators': 400, 'learning_rate': 0.05, 'subsample': 0.8}
  reg_alpha=0.1, reg_lambda=1.0 (same as main comparison notebook)


## 6. Verification: Full-Features Baseline vs Main Table

The "Full (30 feat)" MAE above should match the XGBoost row in the main comparison table.
If it doesn't, something in the pipeline differs.

In [7]:
# Verify by re-training with best params (same as main notebook Cell 17)
print('Verification: re-training Full model with best params...')
lb = bp_xgb['lookback']
X_tr, X_te, y_tr, y_te = build_xgb_data(FEATURES_FULL, lb)

for h in range(HORIZON):
    m = xgb.XGBRegressor(
        max_depth=bp_xgb['max_depth'],
        n_estimators=bp_xgb['n_estimators'],
        learning_rate=bp_xgb['learning_rate'],
        subsample=bp_xgb['subsample'],
        colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:absoluteerror',
        tree_method='hist',
        n_jobs=-1, verbosity=0)
    m.fit(X_tr, y_tr[:, h],
          eval_set=[(X_te, y_te[:, h])], verbose=False)
    mae = mean_absolute_error(y_te[:, h], m.predict(X_te))
    abl_mae = ablation_results['Full (30 feat)'][h]
    match = 'MATCH' if abs(mae - abl_mae) < 0.01 else f'DIFF={mae - abl_mae:+.2f}'
    print(f'  t+{horizon_names[h]}: Verification={mae:.2f}, Ablation Full={abl_mae:.2f} [{match}]')

print(f'\nIf all say MATCH, the ablation baseline is consistent.')
print(f'Compare these numbers with the XGBoost row in Table 7.4/7.6 of the thesis.')

Verification: re-training Full model with best params...
  t+1h: Verification=25.19, Ablation Full=25.19 [MATCH]
  t+2h: Verification=30.28, Ablation Full=30.28 [MATCH]
  t+3h: Verification=36.84, Ablation Full=36.84 [MATCH]

If all say MATCH, the ablation baseline is consistent.
Compare these numbers with the XGBoost row in Table 7.4/7.6 of the thesis.


In [8]:
# Export results
results_df = pd.DataFrame(ablation_results, index=['MAE_1h', 'MAE_2h', 'MAE_3h']).T
results_df.index.name = 'Feature Set'
out_name = f'xgboost_ablation_{PERIOD.lower()}.csv'
results_df.to_csv(out_name, float_format='%.2f')

try:
    from google.colab import files
    files.download(out_name)
except:
    pass

print(f'Saved: {out_name}')
print(results_df.to_string())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: xgboost_ablation_c.csv
                        MAE_1h     MAE_2h     MAE_3h
Feature Set                                         
Full (30 feat)       25.189436  30.278885  36.839550
No ISP (22 feat)     25.588493  31.517128  38.527527
No BM (24 feat)      25.136374  31.576439  36.980370
No ISP+BM (16 feat)  26.140697  32.481972  38.052113
